# P2 — Focal Loss (γ=2) + Class Weight (IndoBERTweet-LoRA)

**Roadmap Macro F1 ≥ 0.80, langkah P2.** P1 (weighted CE) gagal gate (Macro F1 0.7339 < 0.76).
Baseline pembanding (label corrected + kalibrasi w=[1,1.5,1]): acc 0.7746 / Macro F1 0.7394 /
netral R 0.669 / F1 0.601.

**Desain**: sama dengan P1 (konfigurasi trial-4, 5 epoch, `load_best_model_at_end` per val
macro F1) tetapi loss = **Focal Loss**:
FL(p_t) = −α_t·(1−p_t)^γ·log(p_t), dengan **γ=2** dan α_t = class weights
{negatif 0.75, netral 1.32, positif 1.03}.

**Ekspektasi (hipotesis roadmap)**: membantu sampel netral yang sulit → Macro F1 ≥ 0.78.
Gate: Macro F1 ≥ 0.78 → lanjut P3 (kalibrasi ulang); jika ~0.76 → pertimbangkan P4.


In [ ]:
# P0.1 (replikasi): paksa 1 GPU
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))

In [ ]:
# torchao 0.10 tidak kompatibel dengan peft -> uninstall
!pip uninstall -y torchao

In [ ]:
# P0.3 (replikasi): pin stack era 4.x
!pip install --force-reinstall --no-deps "transformers==4.46.3" "peft==0.13.2" "tokenizers==0.20.3" "huggingface-hub==0.26.5"

In [ ]:
# Sel identitas run - WAJIB untuk replikasi (setelah pin versi).
import sys
import torch
import transformers
import peft

assert transformers.__version__.startswith("4.46"), (
    f"transformers {transformers.__version__} bukan pin 4.46 - instalasi bermasalah!"
)
from transformers import TFPreTrainedModel  # bukti tidak ada file campur 5.0

print("python        :", sys.version)
print("torch         :", torch.__version__)
print("transformers  :", transformers.__version__)
print("peft          :", peft.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count     :", torch.cuda.device_count())
print("gpu name      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# =====================================================
# IMPORT LIBRARY
# =====================================================
import os
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
# =====================================================
# SET SEED
# =====================================================
seed = 42
set_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print("GPU tersedia:", torch.cuda.is_available())

In [ ]:
# =====================================================
# LOAD DATA + SPLIT 80:20 (kolom BERT EKSPLISIT: text_bert)
# =====================================================
import os
import pandas as pd
from sklearn.model_selection import train_test_split

print("Isi /kaggle/input:")
mounted = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        p = os.path.join(root, f)
        mounted.append(p)
        print("  ", p)
if not mounted:
    print("  (KOSONG - dataset tidak ter-mount!)")

csv_path = next(
    (p for p in mounted if p.endswith("data_preprocessed_with_emoticon.csv")), None
)
if csv_path is None:
    raise FileNotFoundError(
        "CSV dataset tidak ditemukan di mount. Ter-mount: "
        + (str(mounted) if mounted else "TIDAK ADA APA PUN")
    )
print("CSV ditemukan di:", csv_path)
df = pd.read_csv(csv_path)

if "text_bert" in df.columns:
    col_bert = "text_bert"
else:
    raise ValueError(
        "Kolom 'text_bert' tidak ditemukan di CSV. Kolom tersedia: " + str(df.columns.tolist())
    )
col_label = "label"
print("Kolom BERT terpilih:", col_bert)

df[col_bert] = df[col_bert].fillna("").astype(str)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[col_label]
)

X_train_bert = train_df[col_bert].values
X_test_bert = test_df[col_bert].values
y_train = train_df[col_label].values
y_test = test_df[col_label].values

print(f"Train set: {len(X_train_bert)} | Test set: {len(X_test_bert)}")

In [ ]:
# =====================================================
# SPLIT TRAIN -> TRAIN FINAL + VALIDATION (10%)
# =====================================================
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_bert, y_train, test_size=0.1, stratify=y_train, random_state=42
)

print("Distribusi train final:")
print(pd.Series(y_train_final).value_counts().sort_index())
print()
print("Distribusi validation:")
print(pd.Series(y_val).value_counts().sort_index())
print()
print("Distribusi test:")
print(pd.Series(y_test).value_counts().sort_index())

In [ ]:
# =====================================================
# TOKENIZER
# =====================================================
model_name = "indolem/indobertweet-base-uncased"
tokenizer_bert = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# =====================================================
# DATASET PYTORCH (versi numpy-friendly)
# =====================================================
import torch
from torch.utils.data import Dataset
import numpy as np
import pandas as pd

class SentimenDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.values if isinstance(texts, pd.Series) else np.array(texts)
        self.labels = labels.values if isinstance(labels, pd.Series) else np.array(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [ ]:
# =====================================================
# METRIK EVALUASI (average='macro', zero_division=0)
# =====================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

In [ ]:
# =====================================================
# FUNGSI BUILD INDOBERTWEET-LORA
# =====================================================
def build_indobertweet_lora(dropout=0.3, r=16, lora_alpha=32):
    id2label = {0: "negatif", 1: "netral", 2: "positif"}
    label2id = {"negatif": 0, "netral": 1, "positif": 2}

    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, config=config, ignore_mismatched_sizes=True
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["query", "value"],
        lora_dropout=dropout,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        modules_to_save=["classifier"],
    )
    return get_peft_model(model, lora_config)

In [ ]:
# =====================================================
# BUAT DATASET TRAIN/VAL/TEST
# =====================================================
train_dataset = SentimenDataset(X_train_final, y_train_final, tokenizer_bert, max_length=128)
val_dataset = SentimenDataset(X_val, y_val, tokenizer_bert, max_length=128)
test_dataset = SentimenDataset(X_test_bert, y_test, tokenizer_bert, max_length=128)

print(f"Train {len(train_dataset)} | Val {len(val_dataset)} | Test {len(test_dataset)}")

## P2 — Training Focal Loss (γ=2) + Class Weight

Focal Loss: `FL(p_t) = -alpha_t * (1-p_t)^gamma * log(p_t)`.
γ = 2; alpha_t = class weights {0.75, 1.32, 1.03}.
Custom `FocalLossTrainer` mengganti `compute_loss`.

In [ ]:
# =====================================================
# P2 - FOCAL LOSS (gamma=2) + CLASS WEIGHTS
# =====================================================
class_weights = torch.tensor([0.75, 1.32, 1.03])  # negatif, netral, positif
gamma = 2.0
print("Class weights:", class_weights.tolist(), "| gamma:", gamma)

class FocalLossTrainer(Trainer):
    def __init__(self, *args, gamma=2.0, alpha=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        probs = torch.softmax(logits, dim=-1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
        focal = -(1.0 - pt) ** self.gamma * torch.log(pt.clamp(min=1e-8))
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device).gather(0, labels)
            focal = focal * alpha_t
        loss = focal.mean()
        return (loss, outputs) if return_outputs else loss

best_params = {"batch_size": 16, "dropout": 0.3, "learning_rate": 0.0002, "r": 16, "alpha": 32}

set_seed(seed)
model = build_indobertweet_lora(
    dropout=best_params["dropout"], r=best_params["r"], lora_alpha=best_params["alpha"]
)

training_args = TrainingArguments(
    output_dir="./results_e5_focal",
    learning_rate=best_params["learning_rate"],
    per_device_train_batch_size=best_params["batch_size"],
    per_device_eval_batch_size=best_params["batch_size"],
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    save_total_limit=1,
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    gamma=gamma,
    alpha=class_weights,
)
trainer.train()
eval_result = trainer.evaluate()
print("Hasil validation:", eval_result)

# --- Sanity check (P0): deteksi collapse ---
preds_val = trainer.predict(val_dataset)
y_val_pred = np.argmax(preds_val.predictions, axis=1)
maj = pd.Series(y_val).mode()[0]
p_maj = float((y_val == maj).mean())
f1_maj = (2 * p_maj / (1 + p_maj)) / 3
print("Distribusi prediksi val :", pd.Series(y_val_pred).value_counts().sort_index().to_dict())
print(f"Baseline mayoritas val  : acc={p_maj:.4f} macro_f1={f1_maj:.4f}")
status = "COLLAPSE" if eval_result["eval_f1_macro"] <= f1_maj + 1e-6 else "OK"
print("STATUS:", status)

df_e5_val = pd.DataFrame([{
    "run": "focal_g2",
    "accuracy_val": eval_result["eval_accuracy"],
    "precision_macro_val": eval_result["eval_precision_macro"],
    "recall_macro_val": eval_result["eval_recall_macro"],
    "f1_macro_val": eval_result["eval_f1_macro"],
    "status": status,
}])
print(df_e5_val.to_string(index=False))
df_e5_val.to_csv("hasil_e5_val.csv", index=False)

In [ ]:
# =====================================================
# EVALUASI TEST + SIMPAN PROBABILITAS
# =====================================================
def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

label_names = ["negatif", "netral", "positif"]
mapping = {0: "negatif", 1: "netral", 2: "positif"}

preds_test = trainer.predict(test_dataset)
logits = preds_test.predictions
y_pred_test = np.argmax(logits, axis=1)
P = softmax_np(logits)

print(classification_report(y_test, y_pred_test, target_names=label_names, zero_division=0))
print("Distribusi prediksi:", pd.Series(y_pred_test).value_counts().sort_index().to_dict())
print("Distribusi aktual  :", pd.Series(y_test).value_counts().sort_index().to_dict())

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred_test, average="macro", zero_division=0
)
acc = accuracy_score(y_test, y_pred_test)

hasil = pd.DataFrame({
    "text": pd.Series(X_test_bert),
    "label_aktual": pd.Series(y_test),
    "label_prediksi": pd.Series(y_pred_test),
    "sentimen_prediksi": pd.Series(y_pred_test).map(mapping),
    "prob_negatif": P[:, 0],
    "prob_netral": P[:, 1],
    "prob_positif": P[:, 2],
})
hasil.to_csv("hasil_e5_test.csv", index=False)
print("Tersimpan: hasil_e5_test.csv")

df_e5_test = pd.DataFrame([{
    "run": "focal_g2",
    "accuracy": acc,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
}])
print(df_e5_test.to_string(index=False))
df_e5_test.to_csv("hasil_e5_test_ringkas.csv", index=False)

cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.title("Confusion Matrix - Focal Loss (P2)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# =====================================================
# RINGKASAN AKHIR P2
# =====================================================
print("=== RINGKASAN P2 (focal gamma=2 + CW) ===")
print("Validation:")
print(df_e5_val.to_string(index=False))
print()
print("Test:")
print(df_e5_test.to_string(index=False))
print()
print("McNemar vs baseline dilakukan OFFLINE setelah pull (verify_metrics --compare).")
print("Entri PASCA eksperimen diisi di docs/LOG_EKSPERIMEN.md setelah run selesai.")